In [1]:
!pip install --upgrade --quiet langchain langchain-community langchain-experimental langchain-groq langchain-neo4j neo4j

In [26]:
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.graph_transformers import LLMGraphTransformer
from google.colab import userdata

In [32]:
NEO4J_URI = "your_neo4j_connection_string"
NEO4J_USERNAME = "your_username"
NEO4J_PASSWORD = "your_password"
NEO4J_DATABASE = "your_database"
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)
print("Connected to Neo4j")

Connected to Neo4j


In [31]:
groq_api_key = "your_groq_api_key"
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    groq_api_key=groq_api_key,
    reasoning_effort="low",
    model_kwargs={
        "parallel_tool_calls": False
    }
)

In [5]:
property_texts = [
    "Woodlands Gardens is a 3-bedroom condominium priced at SGD 1200000 with 2 bathrooms. It is located in Woodlands. It is near Woodlands MRT and Woodlands Primary School.",
    "Blk 206 Marsiling Drive is a 3-bedroom HDB 4-room resale flat priced at SGD 580000 with 2 bathrooms. It is located in Marsiling. It is near Marsiling MRT and Woodlands Primary School.",
    "Terrace House at Woodlands Avenue 1 is a 4-bedroom landed property priced at SGD 3200000 with 3 bathrooms. It is located in Woodlands. It is near Woodlands MRT and Woodlands Primary School."
]

documents = [Document(page_content=text.strip()) for text in property_texts]
documents

[Document(metadata={}, page_content='Woodlands Gardens is a 3-bedroom condominium priced at SGD 1200000 with 2 bathrooms. It is located in Woodlands. It is near Woodlands MRT and Woodlands Primary School.'),
 Document(metadata={}, page_content='Blk 206 Marsiling Drive is a 3-bedroom HDB 4-room resale flat priced at SGD 580000 with 2 bathrooms. It is located in Marsiling. It is near Marsiling MRT and Woodlands Primary School.'),
 Document(metadata={}, page_content='Terrace House at Woodlands Avenue 1 is a 4-bedroom landed property priced at SGD 3200000 with 3 bathrooms. It is located in Woodlands. It is near Woodlands MRT and Woodlands Primary School.')]

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = text_splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:", chunk.page_content)


Chunk 1: Woodlands Gardens is a 3-bedroom condominium priced at SGD 1200000 with 2 bathrooms. It is located in Woodlands. It is near Woodlands MRT and Woodlands Primary School.
Chunk 2: Blk 206 Marsiling Drive is a 3-bedroom HDB 4-room resale flat priced at SGD 580000 with 2 bathrooms. It is located in Marsiling. It is near Marsiling MRT and Woodlands Primary School.
Chunk 3: Terrace House at Woodlands Avenue 1 is a 4-bedroom landed property priced at SGD 3200000 with 3 bathrooms. It is located in Woodlands. It is near Woodlands MRT and Woodlands Primary School.


In [9]:
llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Property", "MRTStation", "School", "Neighbourhood"],
    allowed_relationships=["NEAR_MRT", "NEAR_SCHOOL", "LOCATED_IN"],
    node_properties=["price_sgd", "bedrooms", "bathrooms", "property_type"]
)

graph_documents = llm_transformer.convert_to_graph_documents(
    chunks
)

print("Graph documents extracted:", len(graph_documents))
graph_documents = llm_transformer.convert_to_graph_documents(chunks)

Graph documents extracted: 3


In [10]:
graph_documents

[GraphDocument(nodes=[Node(id='Woodlands Gardens', type='Property', properties={'price_sgd': '1200000', 'bedrooms': '3', 'bathrooms': '2', 'property_type': 'condominium'}), Node(id='Woodlands Mrt', type='Mrtstation', properties={}), Node(id='Woodlands Primary School', type='School', properties={}), Node(id='Woodlands', type='Neighbourhood', properties={})], relationships=[Relationship(source=Node(id='Woodlands Gardens', type='Property', properties={}), target=Node(id='Woodlands Mrt', type='Mrtstation', properties={}), type='NEAR_MRT', properties={}), Relationship(source=Node(id='Woodlands Gardens', type='Property', properties={}), target=Node(id='Woodlands Primary School', type='School', properties={}), type='NEAR_SCHOOL', properties={}), Relationship(source=Node(id='Woodlands Gardens', type='Property', properties={}), target=Node(id='Woodlands', type='Neighbourhood', properties={}), type='LOCATED_IN', properties={})], source=Document(metadata={}, page_content='Woodlands Gardens is a 3

In [11]:
for i, graph_doc in enumerate(graph_documents):
    print(f"\n--- Graph Document {i+1} ---")
    print("Nodes:")
    for node in graph_doc.nodes:
        print(node)
    print("Relationships:")
    for rel in graph_doc.relationships:
        print(rel)



--- Graph Document 1 ---
Nodes:
id='Woodlands Gardens' type='Property' properties={'price_sgd': '1200000', 'bedrooms': '3', 'bathrooms': '2', 'property_type': 'condominium'}
id='Woodlands Mrt' type='Mrtstation' properties={}
id='Woodlands Primary School' type='School' properties={}
id='Woodlands' type='Neighbourhood' properties={}
Relationships:
source=Node(id='Woodlands Gardens', type='Property', properties={}) target=Node(id='Woodlands Mrt', type='Mrtstation', properties={}) type='NEAR_MRT' properties={}
source=Node(id='Woodlands Gardens', type='Property', properties={}) target=Node(id='Woodlands Primary School', type='School', properties={}) type='NEAR_SCHOOL' properties={}
source=Node(id='Woodlands Gardens', type='Property', properties={}) target=Node(id='Woodlands', type='Neighbourhood', properties={}) type='LOCATED_IN' properties={}

--- Graph Document 2 ---
Nodes:
id='Blk 206 Marsiling Drive' type='Property' properties={'price_sgd': '580000', 'bedrooms': '3', 'bathrooms': '2', 

In [13]:
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)

print("Graph data added to Neo4j")

Graph data added to Neo4j


In [14]:
graph.refresh_schema()
print(graph.schema)

Node properties:
Property {bedrooms: STRING, bathrooms: STRING, id: STRING, property_type: STRING, price_sgd: STRING}
Document {id: STRING, text: STRING}
Mrtstation {id: STRING}
School {id: STRING}
Neighbourhood {id: STRING}
Relationship properties:

The relationships:
(:Property)-[:NEAR_MRT]->(:Mrtstation)
(:Property)-[:NEAR_SCHOOL]->(:School)
(:Property)-[:LOCATED_IN]->(:Neighbourhood)
(:Document)-[:MENTIONS]->(:Property)
(:Document)-[:MENTIONS]->(:Mrtstation)
(:Document)-[:MENTIONS]->(:School)
(:Document)-[:MENTIONS]->(:Neighbourhood)


In [18]:
result = graph.query("""
MATCH (p:Property)-[r:NEAR_MRT]->(m:Mrtstation)
RETURN
    p.id AS property,
    m.id AS mrt_station,
    type(r) AS relationship
""")

result

ERROR:neo4j.io:[#8894]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-4742ad36-764f-0001.production-orch-1199.neo4j.io', 7687)) (ResolvedIPv4Address(('122.248.207.10', 7687))): ConnectionResetError(104, 'Connection reset by peer')


[{'property': 'Woodlands Gardens',
  'mrt_station': 'Woodlands Mrt',
  'relationship': 'NEAR_MRT'},
 {'property': 'Blk 206 Marsiling Drive',
  'mrt_station': 'Marsiling Mrt',
  'relationship': 'NEAR_MRT'},
 {'property': 'Terrace House At Woodlands Avenue 1',
  'mrt_station': 'Woodlands Mrt',
  'relationship': 'NEAR_MRT'}]

In [19]:
chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True
)


In [20]:
response = chain.invoke({
    "query": "Which properties are near Woodlands MRT?"
})

print(response["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Property)-[:NEAR_MRT]->(m:Mrtstation {id: "Woodlands MRT"})
RETURN p
Full Context:
[]

> Finished chain.
I don’t have that information.


In [21]:
response = chain.invoke({
    "query": "Which properties are located in Woodlands?"
})

print(response["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Property)-[:LOCATED_IN]->(n:Neighbourhood {id: 'Woodlands'})
RETURN p
Full Context:
[{'p': {'bedrooms': '3', 'property_type': 'condominium', 'price_sgd': '1200000', 'id': 'Woodlands Gardens', 'bathrooms': '2'}}, {'p': {'bedrooms': '4', 'property_type': 'landed', 'price_sgd': '3200000', 'id': 'Terrace House At Woodlands Avenue 1', 'bathrooms': '3'}}]

> Finished chain.
Woodlands Gardens, Terrace House At Woodlands Avenue 1.


In [22]:
response = chain.invoke({
    "query": "Which properties are near Woodlands Primary School?"
})

print(response["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (s:School {id: "Woodlands Primary School"})<-[:NEAR_SCHOOL]-(p:Property)
RETURN p.id AS propertyId, p.bedrooms, p.bathrooms, p.property_type, p.price_sgd
Full Context:
[{'propertyId': 'Woodlands Gardens', 'p.bedrooms': '3', 'p.bathrooms': '2', 'p.property_type': 'condominium', 'p.price_sgd': '1200000'}, {'propertyId': 'Blk 206 Marsiling Drive', 'p.bedrooms': '3', 'p.bathrooms': '2', 'p.property_type': 'HDB', 'p.price_sgd': '580000'}, {'propertyId': 'Terrace House At Woodlands Avenue 1', 'p.bedrooms': '4', 'p.bathrooms': '3', 'p.property_type': 'landed', 'p.price_sgd': '3200000'}]

> Finished chain.
I don't know the answer.


In [23]:
response = chain.invoke({
    "query": "Which properties in Woodlands are near Woodlands MRT?"
})

print(response["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Property)-[:LOCATED_IN]->(n:Neighbourhood {id: 'Woodlands'})
MATCH (p)-[:NEAR_MRT]->(m:Mrtstation {id: 'Woodlands'})
RETURN p
Full Context:
[]

> Finished chain.
I don’t have that information.


In [24]:
response = chain.invoke({
    "query": "Which properties are located on the Moon?"
})

print(response["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:

Full Context:
[]

> Finished chain.
I don’t know the answer.
